# AML - SAIGE-GENE SET-BASED ANALYSIS - PLINK FILE FOR VARIANCE RATIO
This script creates the PLINK file used for variance ratio estimation. Note that the file must contain at least 200 variants with MAC values falling within the specified categories. This PLINK file is used only for variance ratio estimation and contains random markers extracted from the LD-pruned PLINK file used for constructing the GRM and performing variance ratio estimation.

In [ ]:
# Load packages 
from datetime import datetime
import pandas as pd 
import os

In [ ]:
# Workspace bucket 
my_bucket = os.getenv("WORKSPACE_BUCKET")
# Workspace bucket 
my_bucket

# COMMAND 

In [ ]:
%%writefile plink_markers.sh
#!/bin/bash

# Exit immediately if a command exits with a non-zero status
set -o errexit

# Treat unset variables as an error (prevents using undefined variables)
set -o nounset

# Loop through the populations
for pop in EUR; do
    echo "Processing population: ${pop}"
    
    # Check if the cohort file for the current population exists
    if [ ! -f "${cohort}/${pop}-like_exome_qc.txt" ]; then
        echo "Cohort file for ${pop} not found!"
        exit 1
    fi
    
    # Copy new plink module into the plink2 
    cp "${plink2}" plink2

    # Grant permision to run plink2 in environment
    chmod +x plink2
    
    # Check number of samples in each cohort 
    echo "Number of samples in ${pop}:"
    wc -l "${cohort}/${pop}-like_exome_qc.txt"
    
    # Extract markers from the large plink file
    ./plink2 --bfile "${genos}/arrays" \
             --keep "${cohort}/${pop}-like_exome_qc.txt" \
             --extract "${markers}/${pop}.markerid.list" \
             --make-bed --out "${out_path}/${pop}_array.forCate_vr"
        
done

In [ ]:
# Copy script to my_bucket 
!gsutil cp {my_bucket}/data/scripts/                                    

In [ ]:
# Copy script to my_bucket 
!gsutil cat {my_bucket}/data/scripts/plink_markers.sh                               

# SUBMIT COMMAND 

In [ ]:
%%bash --out plink_markers 

#############################################################################
# DSUB: SUBMIT SCRIPT FOR VARIANT SUMMARY STATISTICS 
#############################################################################

# Source the dsub setup script, which likely contains environment setup and utility functions
source ~/aou_dsub.bash 

# Define the mount path for input PLINK files
mnt_path="/mnt/data/input/gs/fc-aou-datasets-controlled/v8/microarray/plink"

#############################################################################
# SUBMIT DSUB SCRIPT JOB FOR VARIANT SUMMARY STATISTICS 
#############################################################################

# Use a prebuilt container with PLINK2
# Set total disk space required 
# Define logging directory for dsub execution logs
# Input PLINK files (BED, BIM, FAM) from Google Cloud Storage (GCS)
# Provide cohort sample lists (recursively input from GCS)
# Define output directory for variant QC results (recursively stores outputs)
# Set environment variables for the script
# Specify the script to execute within the container

aou_dsub \
  --disk-size 300 \
  --image ${ARTIFACT_REGISTRY_DOCKER_REPO}/ubuntu:latest \
  --logging "${WORKSPACE_BUCKET}/data/logging" \
  --input plink2="${WORKSPACE_BUCKET}/data/plink2_module/plink2" \
  --input bed="gs://fc-aou-datasets-controlled/v8/microarray/plink/arrays.bed" \
  --input bim="gs://fc-aou-datasets-controlled/v8/microarray/plink/arrays.bim" \
  --input fam="gs://fc-aou-datasets-controlled/v8/microarray/plink/arrays.fam" \
  --input-recursive cohort="${WORKSPACE_BUCKET}/data/wgs_qc/analysis_samples_wgs_qc/exome/" \
  --input-recursive markers="${WORKSPACE_BUCKET}/data/saige_step01/mark_list/" \
  --output-recursive out_path="${WORKSPACE_BUCKET}/data/saige_step01/variance_files/" \
  --env genos="${mnt_path}" \
  --env cohort="${WORKSPACE_BUCKET}/data/wgs_qc/analysis_samples_wgs_qc/exome" \
  --env markers="${WORKSPACE_BUCKET}/data/saige_step01/mark_list" \
  --env out_path="${WORKSPACE_BUCKET}/data/saige_step01/variance_files" \
  --script "${WORKSPACE_BUCKET}/data/scripts/plink_markers.sh"
  
#############################################################################
# END OF SCRIPT 
#############################################################################
